# Table of Contents
* [Section 1: Imports & Configuration](#imports)
* [Section 2: Extracting Words With Their Timecodes](#extract-words)
* [Section 3: Mapping Words To Phonemes](#map-words)
* [Section 4: Rhyme Unit Extraction](#extract-rhyme-units)
* [Section 5: Pairwise Rhyme Similarity](#pairwise-similarity)
* [Section 6: Words Clustering](#clustering)
* [Section 7: HTML Generation](#html-generation)

---
## Section 1: Imports & Configuration <a id="imports"></a>
---

In [1]:
import sys
sys.path.append("..")
print(sys.executable)

from python.rhyme_extraction import (
    extract_rhyme_unit, 
    extract_end_words, 
    extract_rhyme_candidates
)

from python.similarity_engine import (
    rhyme_similarity,
    compute_similarity_pairs,
    build_similarity_matrix
)
from python.clustering import (
    SIMILARITY_THRESHOLD,
    get_threshold,
    cluster_rhymes, 
    cluster_exact_rhymes
)
from python.debugging import debug_rhyme_analysis
from python.evaluation import evaluate_rhyme_detection
from python.html_generation import generate_rhyme_html

from praatio import textgrid

RHYME_MODE = "stressed"
DETECTION_MODE = "full_line"
# Options for RHYME_MODE:   "stressed" / "stressed_plus" / "entire_word"
# Options for DETECTION_MODE: "end_only" / "full_line"

INPUT_TXT_PATH = "input/current_input/input.txt"
OUTPUT_TEXTGRID_PATH = "output/current_output/input.TextGrid"

C:\Users\Georgii\miniconda3\envs\mfa_env\python.exe


---
## Section 2: Extracting Words With Their Timecodes <a id="extract-words"></a>
---

In [2]:
tg = textgrid.openTextgrid(OUTPUT_TEXTGRID_PATH, includeEmptyIntervals=False)

word_tier = tg.getTier("words")
phone_tier = tg.getTier("phones")

for word in word_tier.entries:
    print(word)

Interval(start=1.67, end=1.83, label='i')
Interval(start=1.83, end=2.23, label='found')
Interval(start=2.23, end=2.29, label='the')
Interval(start=2.29, end=2.85, label='sound')
Interval(start=2.91, end=3.1, label='of')
Interval(start=3.1, end=3.26, label='the')
Interval(start=3.26, end=4.03, label='underground')
Interval(start=4.25, end=4.42, label='the')
Interval(start=4.42, end=4.89, label='crowd')
Interval(start=4.89, end=5.13, label='was')
Interval(start=5.13, end=5.57, label='loud')
Interval(start=5.61, end=5.77, label='and')
Interval(start=5.77, end=6.33, label='proud')
Interval(start=6.33, end=6.9, label='around')
Interval(start=6.9, end=7.0, label='the')
Interval(start=7.0, end=7.54, label='mound')
Interval(start=8.08, end=8.16, label='a')
Interval(start=8.16, end=8.53, label='light')
Interval(start=8.66, end=8.86, label='at')
Interval(start=8.86, end=9.24, label='night')
Interval(start=9.24, end=10.01, label='ignites')
Interval(start=10.04, end=10.31, label='the')
Interval(st

---
## Section 3: Mapping Words To Phonemes <a id="map-words"></a>
---

In [3]:
for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    phonemes = []
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    print(word_label + ":", phonemes)

# Populating the word_to_phonemes dictionary
word_to_phonemes = {}

for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    
    phonemes = []
    
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    
    word_label = word_label.lower()
    
    if word_label not in word_to_phonemes:
        word_to_phonemes[word_label] = phonemes

print(word_to_phonemes)

i: ['aj']
found: ['f', 'aw', 'n', 'd']
the: ['ð', 'a']
sound: ['s', 'aw', 'n', 'd']
of: ['ɒ', 'v']
the: ['ð', 'a']
underground: ['ɐ', 'n', 'd', 'ə', 'ɡ', 'ɹ', 'aw', 'n', 'd']
the: ['ð', 'a']
crowd: ['k', 'ɹ', 'aw', 'd']
was: ['ʋ', 'ɒ', 'z']
loud: ['l', 'aw', 'd']
and: ['a', 'n']
proud: ['p', 'ɹ', 'aw', 'd']
around: ['a', 'ɹ', 'aw', 'n', 'd']
the: ['ð', 'a']
mound: ['m', 'aw', 'n', 'd']
a: ['a']
light: ['l', 'aj', 't']
at: ['a', 't']
night: ['n', 'aj', 't']
ignites: ['ɪ', 'ɡ', 'n', 'aj', 't', 's']
the: ['ð', 'a']
kite: ['c', 'iː', 'ʈ', 'ə']
in: ['i', 'n']
flight: ['f', 'l', 'aj', 't']
the: ['ð', 'a']
sight: ['s', 'aj', 't']
of: ['ɒ', 'v']
white: ['ʋ', 'aj', 'ʈ']
delight: ['d', 'ɛ', 'l', 'aj', 't']
shines: ['ʃ', 'aj', 'n', 'z']
bright: ['b', 'ɹ', 'aj', 't']
tonight: ['ʈ', 'ə', 'n', 'aj', 'ʈ']
{'i': ['aj'], 'found': ['f', 'aw', 'n', 'd'], 'the': ['ð', 'a'], 'sound': ['s', 'aw', 'n', 'd'], 'of': ['ɒ', 'v'], 'underground': ['ɐ', 'n', 'd', 'ə', 'ɡ', 'ɹ', 'aw', 'n', 'd'], 'crowd': ['k', 'ɹ', 

---
## Section 4: Rhyme Unit Extraction <a id="extract-rhyme-units"></a>
---

In [4]:
rhyme_candidates = extract_rhyme_candidates(
    INPUT_TXT_PATH,
    word_to_phonemes,
    rhyme_mode=RHYME_MODE,
    detection_mode=DETECTION_MODE
)

for r in rhyme_candidates:
    print(r)

{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 0, 'end_word': 'i', 'phonemes': ['aj'], 'rhyme_unit': ['aj'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 1, 'end_word': 'found', 'phonemes': ['f', 'aw', 'n', 'd'], 'rhyme_unit': ['aw', 'n', 'd'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 2, 'end_word': 'the', 'phonemes': ['ð', 'a'], 'rhyme_unit': ['a'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 3, 'end_word': 'sound', 'phonemes': ['s', 'aw', 'n', 'd'], 'rhyme_unit': ['aw', 'n', 'd'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 4, 'end_word': 'of', 'phonemes': ['ɒ', 'v'], 'rhyme_unit': ['ɒ', 'v'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index'

---
## Section 5: Pairwise Rhyme Similarity <a id="pairwise-similarity"></a>
---

In [5]:
similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)

for pair in similarity_pairs:
    print(pair)

# Building a similarity matrix
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)

for w1 in similarity_matrix:
    print(w1, similarity_matrix[w1])

{'wordA': 'i', 'wordB': 'found', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.625}
{'wordA': 'i', 'wordB': 'sound', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'of', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.625}
{'wordA': 'i', 'wordB': 'underground', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.625}
{'wordA': 'i', 'wordB': 'crowd', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'was', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'loud', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'and', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'proud', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'around', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.625}
{'wordA': 'i', 'wordB': 'mound', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'a', 'similarity': 0.625}
{'wordA': 'i', 'wordB': 'light', 'similarity': 0.75}
{'wordA': 'i', 'wordB': 'at', 'similarity': 0.25}
{'wordA': 'i', 'wordB': 'night', 'similarity': 0.75}
{'word

In [6]:
# Milestone 11 — panphon similarity spot-check
# Rhyme units below are real IPA sequences from english_mfa alignment of the
# project verse (see Section 4 output). They exercise the new feature-based
# scorer on perfect / slant / non-rhyme pairs.
from python.similarity_engine import rhyme_unit_similarity

spot_check_pairs = [
    # Perfect rhymes — expect 1.0
    ("found",  ["aw", "n", "d"],  "sound",  ["aw", "n", "d"]),
    ("light",  ["aj", "t"],       "night",  ["aj", "t"]),
    ("white",  ["aj", "ʈ"],       "tonight", ["aj", "ʈ"]),

    # Insertion slant (-oud vs -ound: one extra 'n') — expect a clear slant, not 0
    ("crowd",  ["aw", "d"],       "mound",  ["aw", "n", "d"]),

    # Substitution slant (t vs retroflex ʈ) — expect high
    ("light",  ["aj", "t"],       "white",  ["aj", "ʈ"]),

    # Genuinely different — expect low
    ("found",  ["aw", "n", "d"],  "light",  ["aj", "t"]),
    ("the",    ["a"],             "mound",  ["aw", "n", "d"]),
]

print(f"{'Word A':<10} {'Word B':<10} {'Score':>6}  Rhyme units")
print("-" * 55)
for wordA, ruA, wordB, ruB in spot_check_pairs:
    score = rhyme_unit_similarity(ruA, ruB)
    print(f"{wordA:<10} {wordB:<10} {score:>6.3f}  {ruA} | {ruB}")


Word A     Word B      Score  Rhyme units
-------------------------------------------------------
found      sound       1.000  ['aw', 'n', 'd'] | ['aw', 'n', 'd']
light      night       1.000  ['aj', 't'] | ['aj', 't']
white      tonight     1.000  ['aj', 'ʈ'] | ['aj', 'ʈ']
crowd      mound       0.812  ['aw', 'd'] | ['aw', 'n', 'd']
light      white       0.833  ['aj', 't'] | ['aj', 'ʈ']
found      light       0.375  ['aw', 'n', 'd'] | ['aj', 't']
the        mound       0.438  ['a'] | ['aw', 'n', 'd']


---
## Section 6: Words Clustering <a id="clustering"></a>
---

In [7]:
# Clustering by thresholds from the similarity matrix
# get_threshold() returns the correct value for the active RHYME_MODE
cluster_labels, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

print("Clusters:")
print(clusters)

print("\nWord → Cluster Mapping:")
print(cluster_labels)

# Defining expected_clusters and comparing with the currect clustering
expected_clusters = {
    "underground": "A",
    "mound": "A",
    "flight": "B",
    "tonight": "B"
}

for r in rhyme_candidates:
    word = r["end_word"]
    r["expected_cluster"] = expected_clusters.get(word, None)

for r in rhyme_candidates:
    print(r)

# Exact rhymes clusters printout
clusters = cluster_labels

print(clusters)

# Debugging the clustering
debug_rhyme_analysis(rhyme_candidates, clusters)

# Accuracy evaluation
accuracy, mismatches = evaluate_rhyme_detection(rhyme_candidates, clusters)

Clusters:
{'A': ['i', 'light', 'night', 'flight', 'sight', 'white', 'delight', 'bright', 'tonight'], 'B': ['found', 'sound', 'underground', 'crowd', 'loud', 'proud', 'around', 'mound'], 'C': ['the', 'a'], 'D': ['of'], 'E': ['was'], 'F': ['and'], 'G': ['at'], 'H': ['ignites'], 'I': ['kite'], 'J': ['in'], 'K': ['shines']}

Word → Cluster Mapping:
{'i': 'A', 'light': 'A', 'night': 'A', 'flight': 'A', 'sight': 'A', 'white': 'A', 'delight': 'A', 'bright': 'A', 'tonight': 'A', 'found': 'B', 'sound': 'B', 'underground': 'B', 'crowd': 'B', 'loud': 'B', 'proud': 'B', 'around': 'B', 'mound': 'B', 'the': 'C', 'a': 'C', 'of': 'D', 'was': 'E', 'and': 'F', 'at': 'G', 'ignites': 'H', 'kite': 'I', 'in': 'J', 'shines': 'K'}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 0, 'end_word': 'i', 'phonemes': ['aj'], 'rhyme_unit': ['aj'], 'is_line_end': False, 'expected_cluster': None}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 1, 

In [8]:
print(clusters)

{'i': 'A', 'light': 'A', 'night': 'A', 'flight': 'A', 'sight': 'A', 'white': 'A', 'delight': 'A', 'bright': 'A', 'tonight': 'A', 'found': 'B', 'sound': 'B', 'underground': 'B', 'crowd': 'B', 'loud': 'B', 'proud': 'B', 'around': 'B', 'mound': 'B', 'the': 'C', 'a': 'C', 'of': 'D', 'was': 'E', 'and': 'F', 'at': 'G', 'ignites': 'H', 'kite': 'I', 'in': 'J', 'shines': 'K'}


In [9]:
# Milestone 9 — full pipeline regression check
from python.similarity_engine import compute_similarity_pairs, build_similarity_matrix
from python.clustering import cluster_rhymes, get_threshold
from python.html_generation import filter_clusters

similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)
word_to_cluster, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

passing_labels = filter_clusters(word_to_cluster, rhyme_candidates)

passing = {}
for entry in rhyme_candidates:
    word = entry["end_word"]
    label = word_to_cluster.get(word)
    if label in passing_labels:
        passing.setdefault(label, [])
        if word not in passing[label]:
            passing[label].append(word)

print("Passing clusters:")
for label, members in sorted(passing.items()):
    print(f"  {label}: {members}")

Passing clusters:
  A: ['i', 'light', 'night', 'flight', 'sight', 'white', 'delight', 'bright', 'tonight']
  B: ['found', 'sound', 'underground', 'crowd', 'loud', 'proud', 'around', 'mound']


---
## Section 7: HTML Generation <a id="html-generation"></a>
---

In [10]:
clusters = clusters = cluster_labels

generate_rhyme_html(rhyme_candidates, clusters, detection_mode=DETECTION_MODE, debug=True)

[HTML] Passing clusters: ['A', 'B']
[HTML] Blocked clusters: ['C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K']
HTML file generated: generated_html/rhyme_visualization.html
